## SfM Bathymetry Using SVR Method

This approach will calculate true depth of SfM point cloud using refractive index directly 

### Getting started

To ensure working environment works properly, set to the top level of repository

In [1]:
cd ..

d:\PROGRAM\PYTHON\sfm-bathy-mapper


### Load Packages

Firstly, we need to import packages that will be use in this notebook

In [2]:
import pandas as pd
import numpy as np


from sfmbathy.prep import load_las,tide_calc
from sfmbathy.process import process_small_angle, export_pc
from sfmbathy.validation import validate_bathymetry
from sfmbathy.svm_prediction import svm_depth_prediction,predict_new_sfm_cloud

### Set Input and Output Directory

Set input and output folder (Please be aware with folder name)

In [3]:
input =r"D:\OneDrive\COMPUTER\S2-MASTER\DATA_TESIS\KEPULAUAN_SERIBU\KELAPA_DUA\LAS_KELAPA_DUA.las"
output = r"D:\OneDrive\COMPUTER\S2-MASTER\DATA_TESIS\KEPULAUAN_SERIBU\KELAPA_DUA\KELAPADUA_PC_SVM_output.las"
train = r"D:\OneDrive\COMPUTER\S2-MASTER\DATA_TESIS\KEPULAUAN_SERIBU\KELAPA_DUA\CHECK_POINT_USV.txt"

## Load Data

The input for this process is the point cloud from SfM Photogrammetry in LAS format.
After loading data, it calculated median coordinate in geographic coordinate (EPSG:4326) for predicting tides model.

In [4]:
pc,las,lon_med,lat_med = load_las(input)
mean_elev= np.mean(pc[:,2])
print(f"Mean elevation : {mean_elev:.3f} meters (Elevation relative to WGS84 ellipsoid)")   

CRS EPSG:32748 → EPSG:4326 | Median Lon: 106.566552, Lat: -5.648954
Mean elevation : 18.068 meters (Elevation relative to WGS84 ellipsoid)


## Set Parameter for Tide Modelling

Predicting tide height for determining current water level based on INATIDES model.

In [5]:
tide_dir=r"D:\tide_models" # Directory where the tide model files are stored
model="INATIDES" # Tide model to use for predictions (e.g., "INATIDES", "TPXO", "FES", etc.)
x = lon_med # Longitude of the location for tide predictions
y = lat_med # Latitude of the location for tide predictions
start_time = "2021-08-29 03:22:00" # In UTC time, Start time for tide predictions (in the format "YYYY-MM-DD HH:MM:SS") 
end_time = "2021-08-29 03:56:00" # In UTC time, End time for tide predictions (in the format "YYYY-MM-DD HH:MM:SS") 
freq = "10min" # Frequency of tide predictions (e.g., '1H' for hourly) and '10min' for every 10 minutes

# Refractive index of water
n_water = "default" # You can specify a custom value for the refractive index of water, or use "default" to use the standard value (approximately 1.33)


In [6]:
wl = tide_calc(tide_dir, model, x, y, start_time, end_time, freq) 

Modelling tides with INATIDES
Location Lon: 106.566552, Lat: -5.648954 | time range: 2021-08-29 03:22:00 to 2021-08-29 03:56:00
Representative Water Level: 0.162 m , refer to MSL
Representative Water Level: 19.086 m , refer to Ellipsoid Reference


### Running depth correction based on SVM Approach

This process will correct point cloud using machine learning with Support Vector Regression (SVR). The training data would be in situ data such as echo sounder survey or bathymetric LiDAR.



In [7]:
result = svm_depth_prediction(
        sfm_points=pc,
        usv_points=train,
        wl=wl,
        max_match_dist=2.0,
        test_size=0.25,
        grid_search=True,
        n_jobs=-1,
        verbose=True,
    )

[svm_depth_prediction] 4433883 underwater points (Z <= 19.0861759185791) used for SVR; 3938 land points (Z > 19.0861759185791) passed through unchanged.
[svm_depth_prediction] Matched 1163 USV points to underwater SfM points (mean match distance = 0.159).
[svm_depth_prediction] Best params from grid search: {'C': 100, 'epsilon': 0.1, 'gamma': 1}
[svm_depth_prediction] Test metrics -> RMSE: 0.1445, MAE: 0.0626, R2: 0.7075


In [8]:
report = result["svm_model"]
report

,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm.If none is given, 'rbf' will be used. If a callable is given it isused to precompute the kernel matrix.For an intuitive visualization of different kernel typessee :ref:`sphx_glr_auto_examples_svm_plot_svm_regression.py`",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",1
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive.The penalty is a squared l2. For an intuitive visualization of theeffects of scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",100
,"epsilon epsilon: float, default=0.1Epsilon in the epsilon-SVR model. It specifies the epsilon-tubewithin which no penalty is associated in the training loss functionwith points predicted within a distance epsilon from the actualvalue. Must be non-negative.",0.1
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False
,"max_iter max_iter: int, default=-1Hard limit on iterations within solver, or -1 for no limit.",-1


In [9]:
pc_corrected = np.array(result["point_cloud_predicted"])
pc_corrected

array([[6.72977120e+05, 9.37538652e+06, 1.75197418e+01, 1.80000000e+01,
        8.90000000e+01, 1.02000000e+02],
       [6.72977540e+05, 9.37538708e+06, 1.75197418e+01, 2.40000000e+01,
        1.04000000e+02, 1.19000000e+02],
       [6.72976610e+05, 9.37538663e+06, 1.75197418e+01, 2.20000000e+01,
        1.05000000e+02, 1.18000000e+02],
       ...,
       [6.73789080e+05, 9.37527060e+06, 1.91200000e+01, 1.62000000e+02,
        1.51000000e+02, 1.52000000e+02],
       [6.73790080e+05, 9.37527212e+06, 1.96200000e+01, 2.30000000e+01,
        3.50000000e+01, 4.50000000e+01],
       [6.73789080e+05, 9.37527193e+06, 1.94500000e+01, 1.04000000e+02,
        1.09000000e+02, 1.09000000e+02]], shape=(4437821, 6))

### Validation Workflow

Set validation parameter :
1. Validation data
2. Tide datum model
3. Point cloud and validation data coordinate system (EPSG)
4. Depth range for validation
5. Output directory


In [10]:
valid_dir = train # Validation file (ASCII: X, Y, Z columns) for bathymetry comparison
datum_model = r"D:\tide_models\DATUM\IS_MSL_ELLIPSOID.tif" # tide datum model (GeoTIFF) for converting ellipsoid heights to MSL heights
pc_epsg = 32749  # EPSG code for the point cloud coordinate system 
min_depth = 0  # Filter: minimum depth (m, positive downward)
max_depth = 15# Filter: maximum depth (m). None = no limit
output_validation = r"D:\OneDrive\COMPUTER\S2-MASTER\DATA_TESIS\KEPULAUAN_SERIBU\KELAPA_DUA\HASIL_UJI"   # Directory to save outputs

In [11]:
results = validate_bathymetry(
    pc_source= pc_corrected, validation_file= valid_dir, datum_geotiff=datum_model, pc_epsg = pc_epsg,
    min_depth= min_depth, max_depth=max_depth,                                  
    max_match_distance=10, msl_reference=0, output_dir=output_validation      
)


SfM BATHYMETRY VALIDATION WORKFLOW

[1/7] Loading data...
Loaded point cloud array: 4437821 points
Loaded validation data: CHECK_POINT_USV.txt | 1163 points

[2/7] Transforming to EPSG:4326...

[3/7] Reading datum offsets...
Read datum offsets: 4437821/4437821 points within GeoTIFF bounds
Read datum offsets: 1163/1163 points within GeoTIFF bounds

[4/7] Converting ellipsoid → MSL...
Converted to MSL: 4437821/4437821 points valid
Converted to MSL: 1163/1163 points valid

[5/7] Spatial matching (nearest neighbor)...
Spatial matching: 1163/1163 validation points matched (threshold: 10 m)

[6/7] Filtering by depth range...
Depth range filter (0-15 m): 0/1163 points retained


ValueError: Too few points after filtering (0). Check depth range and match distance settings.

Export Point Cloud

In [12]:
export_pc(pc_corrected, las, output) 

Corrected LAS file saved to: D:\OneDrive\COMPUTER\S2-MASTER\DATA_TESIS\KEPULAUAN_SERIBU\KELAPA_DUA\KELAPADUA_PC_SVM_output.las
